# 评估智能体（Agents）

我们构建了一个电子邮件助手：它先通过路由器（router）对邮件进行分流与分拣（triage），再将需要回复的邮件交给智能体生成答案。那么，如何确保它在生产环境中也能稳定、正确地工作？这正是测试的价值所在：通过可量化的指标（如回复质量、Token 消耗、延迟、分拣准确率等），为架构决策提供依据。

[LangSmith](https://docs.smith.langchain.com/) 提供了两类主流的测试方式，适合不同阶段与需求：
- 基于常规测试框架（如 Pytest）的测试与记录
- 基于 LangSmith 数据集（Datasets）的批量评测

下文将以电子邮件助手为例，面向大模型技术初学者，逐步展示如何编写测试、如何定义评测集，以及如何用“LLM 作为评审者（LLM-as-Judge）”进行质量打分。

![overview-img](img/overview_eval.png)

#### 加载环境变量

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")

## 如何开展评测

#### Pytest / Vitest

[Pytest](https://docs.pytest.org/en/stable/) 与 Vitest 分别是 Python 与 JavaScript 生态中常用、强大的测试框架。LangSmith 支持与这些框架集成，可在运行测试的同时将结果上报到 LangSmith。本文档示例将使用 Pytest。
- 对已熟悉对应语言/框架的开发者而言，Pytest 上手快、灵活强。
- 当每个用例需要“定制化校验逻辑与通过标准”时（难以抽象为统一评测器），Pytest 尤其合适。

#### LangSmith 数据集（Datasets）

你也可以在 [LangSmith](https://docs.smith.langchain.com/evaluation) 中创建数据集，并用 evaluate API 让助手批量跑数以评测。
- 适合团队协作沉淀“黄金数据集”，持续积累覆盖更多真实场景。
- 可利用生产追踪（traces）、标注队列、合成数据等能力，不断补充新样本。
- 当可以为“所有用例”定义统一评测器（如相似度、精确匹配准确率等）时，效果最佳。

## 测试用例（Test Cases）

测试从“定义用例”开始，这一步既关键也常见难点。本文直接给出一组希望覆盖的邮件示例，并说明要测试的要点。用例定义见 `eval/email_dataset.py`，主要包含：

1. **输入邮件（Input Emails）**：多样化的样本集合。
2. **真实标签（Ground Truth Classifications）**：`Respond`（需要回复）、`Notify`（仅通知/无需回复）、`Ignore`（忽略）。
3. **预期工具调用（Expected Tool Calls）**：对于需要回复的邮件，期望调用到的工具名称。
4. **回复判定标准（Response Criteria）**：判定“好回复”的具体要求（便于自动化评测）。

我们会同时覆盖：
- 端到端“集成”测试：输入邮件 → 智能体 → 最终输出 是否满足“回复判定标准”。
- 工作流关键步骤测试：输入邮件 → 智能体 → 分拣结果 是否匹配“真实标签”。

In [ ]:

%load_ext autoreload
%autoreload 2

from email_assistant.eval.email_dataset import email_inputs, expected_tool_calls, triage_outputs_list, response_criteria_list

test_case_ix = 0

print("Email Input:", email_inputs[test_case_ix])
print("Expected Triage Output:", triage_outputs_list[test_case_ix])
print("Expected Tool Calls:", expected_tool_calls[test_case_ix])
print("Response Criteria:", response_criteria_list[test_case_ix])

## Pytest 示例

下面示范如何用 Pytest 测试工作流中的“工具调用”环节：我们将检查 `email_assistant` 在生成回复的过程中，是否调用了正确的工具。

In [ ]:
import pytest
from email_assistant.eval.email_dataset import email_inputs, expected_tool_calls
from email_assistant.utils import format_messages_string
from email_assistant.email_assistant import email_assistant
from email_assistant.utils import extract_tool_calls

from langsmith import testing as t

@pytest.mark.langsmith
@pytest.mark.parametrize(
    "email_input, expected_calls",
    [   # 选择需要回复的邮件样例
        (email_inputs[0],expected_tool_calls[0]),
        (email_inputs[3],expected_tool_calls[3]),
    ],
)
def test_email_dataset_tool_calls(email_input, expected_calls):
    """验证邮件处理流程中是否包含预期的工具调用。
    
    说明：此测试仅检查是否调用了所有期望的工具，不检查调用顺序，
    也不检查每个工具被调用的次数。如有需要，可单独增加相应检查。
    """
    # 运行邮件助手
    messages = [{"role": "user", "content": str(email_input)}]
    result = email_assistant.invoke({"messages": messages})
            
    # 从消息列表中提取工具调用
    extracted_tool_calls = extract_tool_calls(result['messages'])
            
    # 检查是否缺少任何期望的工具调用
    missing_calls = [call for call in expected_calls if call.lower() not in extracted_tool_calls]
    
    t.log_outputs({
                "missing_calls": missing_calls,
                "extracted_tool_calls": extracted_tool_calls,
                "response": format_messages_string(result['messages'])
            })

    # 若无缺失的期望调用，则测试通过
    assert len(missing_calls) == 0

可以注意到以下要点：
- 如需[使用 Pytest 并将结果上报到 LangSmith](https://docs.smith.langchain.com/evaluation/how_to_guides/pytest)，只需在测试函数上添加 `@pytest.mark.langsmith` 装饰器，并将其置于 `.py` 文件中（示例见 `notebooks/test_tools.py`）。这样即可将测试结果自动记录到 LangSmith。
- 还可以通过 `@pytest.mark.parametrize` 传入数据集样例，实现参数化测试（示例见[官方文档](https://docs.smith.langchain.com/evaluation/how_to_guides/pytest#parametrize-with-pytestmarkparametrize)）。

#### 运行 Pytest
可直接在命令行运行测试。我们已将上述代码写入 Python 文件，在项目根目录执行：

`! LANGSMITH_TEST_SUITE='Email assistant: Test Tools For Interrupt'  pytest notebooks/test_tools.py`

#### 查看实验结果

可以在 LangSmith UI 中看到测试记录：
- 断言 `assert len(missing_calls) == 0` 的通过/失败会显示在 `Pass` 列。
- 通过 `t.log_outputs(...)` 上报的数据会显示在 `Outputs` 列；函数入参与上下文信息会显示在 `Inputs` 列。
- 通过 `@pytest.mark.parametrize(...)` 提供的每组输入，都会在 `LANGSMITH_TEST_SUITE` 指定的项目下形成单独一行记录（位于 `Datasets & Experiments`）。

![Test Results](img/test_result.png)

## LangSmith 数据集示例

![overview-img](img/eval_detail.png)

接下来演示如何使用 LangSmith 数据集进行评测。在前面的 Pytest 示例中，我们测试了“工具调用是否正确”。本节将针对“邮件是否需要回复”的分拣（triage）步骤，构建并运行数据集级评测。

#### 数据集定义 

可以通过 LangSmith SDK [创建数据集](https://docs.smith.langchain.com/evaluation/how_to_guides/manage_datasets_programmatically#create-a-dataset)。下面的代码会用 `eval/email_dataset.py` 中的示例，创建一个用于分拣评测的数据集。

In [ ]:
from langsmith import Client

from email_assistant.eval.email_dataset import examples_triage

# 初始化 LangSmith 客户端
client = Client()

# 数据集名称
dataset_name = "E-mail Triage Evaluation"

# 若数据集不存在则创建
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(
        dataset_name=dataset_name, 
        description="A dataset of e-mails and their triage decisions."
    )
    # 将样例加入数据集
    client.create_examples(dataset_id=dataset.id, examples=examples_triage)

#### 目标函数（Target Function）

数据集的基本结构如下：输入是一封邮件，输出是该邮件的“真实分拣标签（是否需要回复）”。

```
examples_triage = [
  {
      "inputs": {"email_input": email_input_1},
      "outputs": {"classification": triage_output_1},   # 说明：在创建数据集时，该项成为 reference_output
  }, ...
]
```

In [ ]:
print("Dataset Example Input (inputs):", examples_triage[0]['inputs'])

In [ ]:
print("Dataset Example Reference Output (reference_outputs):", examples_triage[0]['outputs'])

我们定义一个函数，接收数据集的 `inputs`，并将其传入邮件助手。LangSmith 的 [evaluate API](https://docs.smith.langchain.com/evaluation) 会把数据集中的 `inputs` 字典传给该函数；函数返回的字典即为智能体输出。本节仅评测“分拣”步骤，因此只需返回分拣决策结果。 

In [ ]:
def target_email_assistant(inputs: dict) -> dict:
    """基于工作流的邮件助手：仅执行分拣节点并返回分拣结果。"""
    response = email_assistant.nodes['triage_router'].invoke({"email_input": inputs["email_input"]})
    return {"classification_decision": response.update['classification_decision']}

#### 评测器函数（Evaluator Function）

接下来编写评测器函数。目标很直接：比较“智能体输出”与“数据集参考输出”。

- 参考输出（reference_outputs）：`{"classification": triage_output_1}`
- 智能体输出（outputs）：`{"classification_decision": agent_output_1}`

我们只需对二者进行对比判等即可：当 `outputs` 与 `reference_outputs` 中对应字段一致时，返回通过；否则返回不通过。

In [ ]:
def classification_evaluator(outputs: dict, reference_outputs: dict) -> bool:
    """检查模型分拣结果是否与参考答案完全一致（大小写不敏感）。"""
    return outputs["classification_decision"].lower() == reference_outputs["classification"].lower()

### 运行评测

这些组件如何串在一起？evaluate API 会自动完成：
- 将数据集中的 `inputs` 传入目标函数（target function）。
- 将数据集中的 `reference_outputs` 与目标函数返回的 `outputs` 一并传给评测器函数。

这与我们在 Pytest 中的做法类似：Pytest 用 `@pytest.mark.parametrize` 将“样例输入与参考输出”传入测试函数。

In [ ]:
# 若需启动评测，将其置为 True
run_expt = True
if run_expt:
    experiment_results_workflow = client.evaluate(
        # 评测目标函数（运行智能体指定步骤）
        target_email_assistant,
        # 数据集名称
        data=dataset_name,
        # 评测器列表
        evaluators=[classification_evaluator],
        # 实验名称前缀
        experiment_prefix="E-mail assistant workflow", 
        # 并发评测数量
        max_concurrency=2, 
    )

可以在 LangSmith UI 中查看上述两种评测方式的结果。

![Test Results](img/eval.png)

## LLM 作为评审者（LLM-as-Judge）

前文通过 evaluate() 与 Pytest 展示了“分拣单元测试”和“工具调用测试”。

本节演示如何让 LLM 作为评审者，对智能体的“完整回复”按既定成功标准进行主观评价（结构化打分）。

![types](img/eval_types.png)

首先，定义一个结构化输出的 Pydantic 模型，包含“是否满足标准（布尔）”与“判定理由（文本）”。

In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class CriteriaGrade(BaseModel):
    """依据给定标准对助手回复进行评分。"""
    justification: str = Field(description="对打分与结论的详细说明，请引用回复中的具体片段作为依据。")
    grade: bool = Field(description="回复是否满足给定的成功标准？")
    
# 复用同一个评测用 LLM，避免每次测试重复创建
criteria_eval_llm = init_chat_model("openai:gpt-4o")
criteria_eval_structured_llm = criteria_eval_llm.with_structured_output(CriteriaGrade)

In [ ]:
email_input = email_inputs[0]
print("Email Input:", email_input)
success_criteria = response_criteria_list[0]
print("Success Criteria:", success_criteria)

调用邮件助手获取完整回复，并格式化为字符串；然后与“成功标准”一并输入给 LLM 评审器，得到布尔评分与判定理由。

In [ ]:
response = email_assistant.invoke({"email_input": email_input})

In [ ]:
from email_assistant.eval.prompts import RESPONSE_CRITERIA_SYSTEM_PROMPT

all_messages_str = format_messages_string(response['messages'])
eval_result = criteria_eval_structured_llm.invoke([
        {"role": "system",
            "content": RESPONSE_CRITERIA_SYSTEM_PROMPT},
        {"role": "user",
            "content": f"""\n\n Response criteria: {success_criteria} \n\n Assistant's response: \n\n {all_messages_str} \n\n Evaluate whether the assistant's response meets the criteria and provide justification for your evaluation."""}
    ])

eval_result

In [ ]:
RESPONSE_CRITERIA_SYSTEM_PROMPT

可以看到，评审结果的结构与我们定义的 `CriteriaGrade` 模型一致。

## 面向更大测试集的评测
在已掌握 Pytest 与 evaluate() 的基础上，并结合 LLM 评审方法，我们可以用更大规模的测试集来全面了解助手在不同场景下的表现。

运行更大规模的测试集：
```
! LANGSMITH_TEST_SUITE='Email assistant: Test Full Response Interrupt' LANGSMITH_EXPERIMENT='email_assistant' pytest tests/test_response.py --agent-module email_assistant
```

在 `test_response.py` 中，你会看到：

将数据集样例传入测试函数，并上报到 `LANGSMITH_TEST_SUITE`：
```
# 参考输出的键名
@pytest.mark.langsmith(output_keys=["criteria"])
# 变量名与测试用例列表
# 每个用例为 (email_input, email_name, criteria, expected_calls)
@pytest.mark.parametrize("email_input,email_name,criteria,expected_calls",create_response_test_cases())
def test_response_criteria_evaluation(email_input, email_name, criteria, expected_calls):
```

使用 LLM-as-judge 的评分结构：
```
class CriteriaGrade(BaseModel):
    """依据给定标准对助手回复进行评分。"""
    grade: bool = Field(description="回复是否满足给定的成功标准？")
    justification: str = Field(description="对打分与结论的详细说明，请引用回复中的具体片段作为依据。")
```

依据标准对助手回复进行评估：
```
    # 基于标准进行评估
    eval_result = criteria_eval_structured_llm.invoke([
        {"role": "system",
            "content": RESPONSE_CRITERIA_SYSTEM_PROMPT},
        {"role": "user",
            "content": f"""\n\n Response criteria: {criteria} \n\n Assistant's response: \n\n {all_messages_str} \n\n Evaluate whether the assistant's response meets the criteria and provide justification for your evaluation."""}
    ])
```

现在可以在 LangSmith UI 中查看这次实验的表现，观察系统的优点与可改进之处。

#### 获取结果

也可以通过读取与实验绑定的追踪项目（tracing project）来获取评测结果，便于生成自定义的可视化报表与指标统计。

In [ ]:
# TODO: 在此处填入你的实验名称
experiment_name = "email_assistant:8286b3b8"
# 是否加载实验结果
load_expt = False
if load_expt:
    email_assistant_experiment_results = client.read_project(project_name=experiment_name, include_stats=True)
    print("延迟 p50:", email_assistant_experiment_results.latency_p50)
    print("延迟 p99:", email_assistant_experiment_results.latency_p99)
    print("Token 用量:", email_assistant_experiment_results.total_tokens)
    print("反馈统计:", email_assistant_experiment_results.feedback_stats)